# Trainable inducing locations · 11-input angular GP
One nominal GPU fit per run. Change `M` in the first code cell to choose the number of k-means centres. GPflow optimizes all their coordinates jointly with the kernel, mean and likelihood parameters. The target is the cosine between the true B and true kaon in the CM frame. The predicted angle and reconstructed kaon kinematics give the validation $q^2$ estimate.

In [ ]:
from pathlib import Path
import json
from swan_runner import ROOT, create_session, training_env, preflight, fit, run_logged

M = 1000
EXISTING_WORKSPACE = None  # Optional: previous /tmp/swan-gpflow-... folder on this worker
REFERENCE = None  # Optional path to a completed trainable M=300 fit

if type(M) is not int or not 1 <= M <= 5000:
    raise ValueError("Choose an integer M between 1 and 5000")
session = create_session()
out = session / f"trainable11_M{M}"
print("Fit output:", out)
python, env = training_env(session, isolated=True, existing_workspace=EXISTING_WORKSPACE)

## Verify the GPU
Checks double-precision Cholesky placement and finite gradients for a small trainable SGPR.

In [ ]:
preflight(python, env, session)

## Fit the angular GP
The optimizer runs for up to 1,000 iterations. Validation every 25 iterations, plus the initial and final iterations, determines the saved checkpoint. Each setup-cell execution creates a fresh run directory.

In [ ]:
fit(python, env, out, inducing=M)

In [ ]:
report = json.loads((out / "result.json").read_text())
print(f"M = {report['inducing_points']} | {report['training_rows']} training | {report['validation_rows']} validation")
print(f"Selected step = {report['best_validation_step']}, validation q² RMSE = {report['best_validation_q2_rmse']:.6f} GeV²")
print(f"Inducing-coordinate RMS displacement at selected step = {report['selected_Z_rms_displacement']:.6f} (standardized units)")
print(f"Optimizer: {report['optimizer_message']}, fit time = {report['seconds']/60:.1f} min")

## Compare the GP with Belle II inclusive tagging

Belle II quotes an inclusive-tag $q^2$ resolution of **approximately $1.0\ \mathrm{GeV}^2$**. The quoted passage gives an approximate resolution without specifying a width convention or uncertainty. The table keeps it in a dedicated column. [Belle II, Phys. Rev. D **112**, 092016 (2025), Sec. II](https://digital.library.adelaide.edu.au/server/api/core/bitstreams/68ddbb63-de62-470a-b41b-5afeb4c421fa/content)

For the paired toy comparison, apply Belle II's published estimator $q^2_{\rm ITA}=s/4+m_K^2-\sqrt{s}\,E_K^*$ to the reconstructed kaon on the exact events scored by the GP. Both toy rows use $r=q^2_{\rm rec}-q^2_{\rm true}$. The resolution measures are the standard deviation of $r$, the half-width between its 16th and 84th percentiles, and RMSE including bias. The quoted percentage gain compares the **two toy RMSEs**. The toy uses a simplified response and a truth-designated kaon. Belle II applies candidate selection and multivariate background rejection. [Belle II, Phys. Rev. D **109**, 112006 (2024), Eq. (2)](https://doi.org/10.1103/PhysRevD.109.112006)

Run this block directly on saved predictions. Validation is the default during the scan. To use the frozen model on test, first run the optional test evaluation below and change `SPLIT` to `"test"`. In a fresh kernel, open the notebook from the extracted project and replace `out` with the path to your completed fit directory.

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

RUN = Path(out)  # Or Path("/eos/.../runs/.../trainable11_M1000") for an existing fit
SPLIT = "validation"
DATA_FILE = Path("data/eleven.npz")  # Relative to the extracted notebook directory
split_settings = {"validation": (1, "q2"), "test": (2, "q2_gp")}
split_id, prediction_key = split_settings[SPLIT]
pred_file = RUN / f"{SPLIT}_predictions.npz"
if not pred_file.is_file():
    raise FileNotFoundError(f"Missing {pred_file}. Complete the fit or run the optional test evaluation.")
fit_info = json.loads((RUN / "result.json").read_text())
if hashlib.sha256(DATA_FILE.read_bytes()).hexdigest() != fit_info["input_sha256"]:
    raise ValueError("The dataset differs from the one used for this fit.")

with np.load(DATA_FILE, allow_pickle=False) as data, np.load(pred_file, allow_pickle=False) as pred:
    take = data["split"] == split_id
    # The prediction writer preserves split order. Verify every event before comparing.
    for key in ("generator_seed", "event_id", "q2_true"):
        np.testing.assert_array_equal(data[key][take], pred[key])
    truth = data["q2_true"][take]
    gp = pred[prediction_key]
    gen = json.loads(str(data["metadata_json"]))["generator"]
    sqrt_s = gen["sqrt_s_GeV"]
    energy = data["X"][take, data["feature_names"].tolist().index("k_E_cm")]
    ita = sqrt_s**2 / 4 + gen["mK_GeV"]**2 - sqrt_s * energy
    np.testing.assert_allclose(ita, data["q2_energy_belle"][take], atol=1e-12, rtol=0)

residuals = {"Belle II ITA estimator on toy": ita - truth, "Angular GP on toy": gp - truth}
def summarize(r):
    q16, q84 = np.quantile(r, [0.16, 0.84])
    return dict(bias=float(r.mean()), sigma=float(r.std()), half68=float((q84-q16)/2),
                rmse=float(np.sqrt(np.mean(r**2))))

metrics = {name: summarize(r) for name, r in residuals.items()}
table = ["| Sample / method | Published resolution | Bias | Residual SD | Central 68% half-width | RMSE |",
         "|---|---:|---:|---:|---:|---:|",
         "| Belle II inclusive tag (published) | ≈1.0 | — | — | — | — |"]
for name, row in metrics.items():
    table.append(f"| {name} | — | {row['bias']:+.4f} | {row['sigma']:.4f} | {row['half68']:.4f} | {row['rmse']:.4f} |")
display(Markdown(f"**M = {fit_info['inducing_points']}, {SPLIT}, {len(truth):,} toy events. All values in GeV².**"
                 + "\n\n" + "\n".join(table)))

base2, gp2 = (r**2 for r in residuals.values())
def gain(a, b):
    return 100 * (1 - np.sqrt(b.mean()/a.mean()))
rng = np.random.default_rng(120)
boot = []
for _ in range(2500):
    idx = rng.integers(len(truth), size=len(truth))
    boot.append(gain(base2[idx], gp2[idx]))
lo, hi = np.quantile(boot, [0.025, 0.975])
print(f"Toy RMSE reduction vs ITA estimator = {gain(base2, gp2):.2f}% (paired event-bootstrap 95% interval {lo:.2f}% to {hi:.2f}%)")

fig, ax = plt.subplots(figsize=(7, 3.6))
edges = np.histogram_bin_edges(np.concatenate(list(residuals.values())), bins=70)
for name, residual in residuals.items():
    ax.hist(residual, bins=edges, density=True, histtype="step", linewidth=1.7, label=name)
ax.axvline(0, color="0.5", linewidth=0.8)
ax.set(xlabel="Reconstructed − true q² [GeV²]", ylabel="Probability density [GeV⁻²]",
       title=f"Signal-only toy · {SPLIT} · M = {fit_info['inducing_points']}")
ax.legend()
fig.tight_layout()
fig.savefig(RUN / f"ita_resolution_{SPLIT}.png", dpi=160)
plt.show()

summary = dict(split=SPLIT, n=len(truth), inducing_points=fit_info["inducing_points"], units="GeV²",
               published_ita_approx_resolution=1.0, published_reference="Phys. Rev. D 112, 092016 (2025), Sec. II",
               estimator_reference="Phys. Rev. D 109, 112006 (2024), Eq. (2)", toy=metrics,
               toy_rmse_reduction_percent=gain(base2, gp2), paired_95_percent_interval=[lo, hi])
(RUN / f"ita_resolution_{SPLIT}.json").write_text(json.dumps(summary, indent=2) + "\n")

## Compare with the earlier trainable M=300 fit
To compare inducing counts, set `REFERENCE` to the previous `trainable11` fit directory and rerun just this comparison. Negative delta means lower RMSE for the current fit. The interval resamples paired validation events conditional on the saved fits.

In [ ]:
if REFERENCE is not None and (Path(REFERENCE) / "result.json").is_file():
    run_logged([python, ROOT / "compare_m.py", REFERENCE, out], session / "comparison.log", env)
else:
    print("Comparison skipped. Set REFERENCE to your earlier trainable11 run directory to enable it.")

## Optional test evaluation
Enable after choosing M on validation.

In [ ]:
SCORE_TEST = False
if SCORE_TEST:
    run_logged([python, ROOT / "score.py", out], session / "score.log", env)